# 加载模型并打印模型信息

In [1]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

print("正在加载模型（首次运行会下载约 500MB）...")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()

正在加载模型（首次运行会下载约 500MB）...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

# 代码实验一：亲眼看"下一个词预测"

In [ ]:
import torch

prompt = "Thank you very"

input_ids = tokenizer.encode(prompt, return_tensors="pt")  # 文字 → token ID
tokens = [tokenizer.decode(id) for id in input_ids[0]]
logits = model(input_ids).logits[0, -1, :]   # 模型输出 50257 个原始得分
probs = torch.softmax(logits, dim=0)          # 得分 → 概率
next_token = torch.argmax(probs)               # 选概率最高的

# 模型认为最可能的topK个词
top_k = 10
top_probs, top_indices = torch.topk(probs, top_k)
print(f"\n模型预测 '{prompt}' 后面最可能的 {top_k} 个词：")
print("-" * 50)
for i in range(top_k):
    token = tokenizer.decode(top_indices[i])
    prob = top_probs[i].item() * 100
    bar = "█" * int(prob / 2)
    print(f"  {i+1:2d}. '{token}' \t {prob:5.1f}%  {bar}")

print(f"Token IDs: {input_ids.tolist()[0]}")
print(f"tokens: {tokens}")
print(f"Token logits: {logits.tolist()}")
print(f"Token probs: {probs.tolist()}")
print(f"next token: {next_token}")

Token IDs: [10449, 345, 845]
tokens: ['Thank', ' you', ' very']
Token logits: [-59.0982666015625, -58.95176696777344, -67.32730102539062, -66.21736907958984, -62.659698486328125, -65.25647735595703, -61.084163665771484, -63.3232421875, -60.68052673339844, -63.85926055908203, -64.84701538085938, -51.405303955078125, -59.10016632080078, -57.004798889160156, -61.401023864746094, -63.26791763305664, -61.99852752685547, -62.10474395751953, -61.33221435546875, -62.45569610595703, -61.03236389160156, -63.30077362060547, -61.900482177734375, -64.512451171875, -63.546993255615234, -60.267799377441406, -59.472049713134766, -62.15648651123047, -62.783363342285156, -62.6234130859375, -60.86375427246094, -64.21675109863281, -66.06979370117188, -64.97822570800781, -65.47821044921875, -66.57853698730469, -64.92167663574219, -65.97822570800781, -65.19988250732422, -63.813201904296875, -64.73014068603516, -65.52345275878906, -63.76736831665039, -67.52940368652344, -62.75669860839844, -65.40646362304688